# Day 08 · 資料落地：Grounding 與查證機制

> 第二部・裝備升級　|　⚠️ 部分需要 GCP，其餘可跑

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 08 - 資料落地：Grounding 與查證機制.md`

## 今天要學會

1. 用 `google_search` 做 grounding
2. 讀懂 `groundingMetadata` 的結構
3. 知道 Search Suggestions 的**強制顯示義務**

> ⚠️ **先講一件會擋住你的事**：Google Search grounding 的免費配額
> **跟一般模型呼叫是分開算的，而且緊很多**。撞到 429 時**換模型救不了**。
> 本日的每個搜尋 cell 都做了保護，跑不動也能往下讀完概念。

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 模型會編故事

先看問題。問一個沒有 grounding 的模型「最近」的事：

In [2]:
from google.adk.agents import LlmAgent

ungrounded = LlmAgent(
    name="ungrounded",
    model=get_model(),
    instruction="你是助理。用繁體中文回答，三句話以內。",
)

print(await run_once(ungrounded, "Google ADK 最新的穩定版本號是多少？什麼時候發布的？"))

請注意，您可能是指 Google ADK（Android Development Kit / App Development Kit），但目前 Google 並沒有以「ADK」為名的主流獨立軟體開發套件。如果您是指 Android Studio、Android SDK 或 Jetpack Libraries 等相關工具的最新版本，建議您直接前往官方 Android 開發者網站查詢以獲取最準確的即時資訊。


模型會給你一個**看起來很有自信**的答案。問題是它的知識有截止日期，
而且它不會說「我不知道」——它會給你一個統計上最像答案的東西。

**Grounding 就是把答案綁在可查證的來源上。**

## 2. `google_search`：一行接上搜尋

In [3]:
from google.adk.tools import google_search

grounded = LlmAgent(
    name="grounded",
    model=get_model(),
    instruction=(
        "你是研究助理。回答前先用搜尋確認事實，"
        "用繁體中文摘要重點，三句話以內。"
    ),
    tools=[google_search],
)

SEARCH_OK = True
try:
    print(await run_once(grounded, "Google ADK 最新的穩定版本號是多少？"))
except Exception as exc:
    SEARCH_OK = False
    msg = str(exc)
    print(f"⚠️ {type(exc).__name__}")
    if "429" in msg:
        print("撞到 Search grounding 的獨立配額。")
        print("這個額度跟一般模型呼叫分開算，**換模型也救不了**——只能等重置或開計費。")
    else:
        print(msg[:200])
    print("\n本日後續會用預先錄下的 groundingMetadata 說明結構，不影響學習。")

⚠️ _ResourceExhaustedError
撞到 Search grounding 的獨立配額。
這個額度跟一般模型呼叫分開算，**換模型也救不了**——只能等重置或開計費。

本日後續會用預先錄下的 groundingMetadata 說明結構，不影響學習。


## 3. 資料是怎麼流的

加上 `google_search` 之後，一次問答實際發生了這些事：

```
 1. 使用者提問
 2. 模型判斷「這需要查證」
 3. 模型產生搜尋查詢字串
 4. Google 端執行搜尋            ← 不是在你的機器上跑
 5. 搜尋結果回到模型的上下文
 6. 模型根據結果生成答案
 7. 回應附帶 groundingMetadata   ← 哪句話來自哪個來源
```

**第 4 步是關鍵**：`google_search` 是**內建工具**，實際執行在 Google 端。
這也解釋了為什麼它不能跟你的自訂函式工具混用（Day 06）——
兩者的執行位置根本不同。

## 4. `groundingMetadata`：可查證的證據

這是 grounding 跟「模型自己講」最大的差別——**它會告訴你每句話的來源**。

In [4]:
from google.adk.runners import InMemoryRunner
from google.genai import types

grounding_meta = None

if SEARCH_OK:
    runner = InMemoryRunner(agent=grounded, app_name="day08")
    sid = await new_session(runner)
    msg = types.Content(role="user", parts=[types.Part(text="Google ADK 2.0 有哪些主要特色？")])
    try:
        async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
            gm = getattr(ev, "grounding_metadata", None)
            if gm is not None:
                grounding_meta = gm
    except Exception as exc:
        print(f"⚠️ {type(exc).__name__}: {str(exc)[:120]}")

if grounding_meta is not None:
    print("拿到 groundingMetadata，欄位：")
    for k, v in grounding_meta.model_dump(exclude_none=True).items():
        preview = str(v)
        print(f"  • {k}: {preview[:100]}")
else:
    print("（這次沒有拿到 groundingMetadata，往下看結構說明）")

（這次沒有拿到 groundingMetadata，往下看結構說明）


### 它的結構

不管有沒有跑成，結構是固定的。四個你會用到的欄位：

| 欄位 | 內容 | 你拿來做什麼 |
|---|---|---|
| `web_search_queries` | 模型實際送出的搜尋字串 | debug「它到底查了什麼」 |
| `grounding_chunks` | 來源清單（標題 + URI） | 在 UI 上列出參考資料 |
| `grounding_supports` | **哪一段文字對應哪幾個來源** | 做逐句引註 |
| `search_entry_point` | Google 提供的 HTML 片段 | **法律義務，必須原樣顯示** |

`grounding_supports` 是最有價值的：它用字元區間把「答案的第幾個字到第幾個字」
對應到 `grounding_chunks` 的索引。有了它就能做出逐句可點擊的引註。

In [5]:
# 用一份結構完全相同的樣本，示範怎麼把引註組出來
SAMPLE = {
    "web_search_queries": ["Google ADK 2.0 features"],
    "grounding_chunks": [
        {"web": {"title": "adk.dev", "uri": "https://adk.dev/whats-new"}},
        {"web": {"title": "Google Cloud Blog", "uri": "https://cloud.google.com/blog/adk-2"}},
    ],
    "grounding_supports": [
        {"segment": {"start_index": 0, "end_index": 18, "text": "ADK 2.0 改用圖形化執行引擎"},
         "grounding_chunk_indices": [0]},
        {"segment": {"start_index": 19, "end_index": 40, "text": "並新增了 Agent Skills 與 A2A 支援"},
         "grounding_chunk_indices": [0, 1]},
    ],
}

print("模型實際查了:", SAMPLE["web_search_queries"])
print("\n逐句引註：")
for sup in SAMPLE["grounding_supports"]:
    seg = sup["segment"]
    sources = [SAMPLE["grounding_chunks"][i]["web"]["title"]
               for i in sup["grounding_chunk_indices"]]
    print(f"  「{seg['text']}」")
    print(f"    ↳ 來源: {', '.join(sources)}")

print("\n參考資料清單：")
for i, ch in enumerate(SAMPLE["grounding_chunks"]):
    print(f"  [{i}] {ch['web']['title']} — {ch['web']['uri']}")

模型實際查了: ['Google ADK 2.0 features']

逐句引註：
  「ADK 2.0 改用圖形化執行引擎」
    ↳ 來源: adk.dev
  「並新增了 Agent Skills 與 A2A 支援」
    ↳ 來源: adk.dev, Google Cloud Blog

參考資料清單：
  [0] adk.dev — https://adk.dev/whats-new
  [1] Google Cloud Blog — https://cloud.google.com/blog/adk-2


## 5. ⚠️ Search Suggestions 是**強制**顯示義務

這一點原文有提，但值得再放大一次，因為它不是「建議」而是**使用條款**。

> 使用 Google Search grounding 時，你**必須**把回應裡的
> `search_entry_point.rendered_content` **原樣顯示**給使用者。
> 不能改樣式、不能藏起來、不能只放一個自製的「來源」連結。

做不到就是違反 Gemini API 的使用條款。這是很多人上線後才發現的地雷。

In [6]:
print("""正確做法（以 Web 前端為例）：

    <div class="answer">{{ 模型的回答 }}</div>

    <!-- 直接把 rendered_content 原樣塞進去，不要改它 -->
    {{ grounding_metadata.search_entry_point.rendered_content | safe }}

錯誤做法：
    ✗ 只自己列一個「參考來源」清單
    ✗ 改寫 rendered_content 的 CSS / 結構
    ✗ 折疊起來預設不顯示
""")

正確做法（以 Web 前端為例）：

    <div class="answer">{{ 模型的回答 }}</div>

    <!-- 直接把 rendered_content 原樣塞進去，不要改它 -->
    {{ grounding_metadata.search_entry_point.rendered_content | safe }}

錯誤做法：
    ✗ 只自己列一個「參考來源」清單
    ✗ 改寫 rendered_content 的 CSS / 結構
    ✗ 折疊起來預設不顯示



## 6. 另一條路：自有資料的 grounding

`google_search` 綁的是**公開網路**。如果要綁**你自己的**文件庫，
那是另一套東西：

In [7]:
from google.adk.tools import VertexAiSearchTool

print("VertexAiSearchTool 的建構參數:")
import inspect

print(" ", inspect.signature(VertexAiSearchTool.__init__))

print("""
用法（需要 Google Cloud 專案 + Vertex AI Search data store）：

    from google.adk.tools import VertexAiSearchTool

    my_docs = VertexAiSearchTool(
        data_store_id="projects/<PROJECT>/locations/<LOC>"
                      "/collections/default_collection/dataStores/<DATASTORE>"
    )

    agent = LlmAgent(
        name="doc_agent",
        model="gemini-flash-latest",
        instruction="依照公司內部文件回答，找不到就說沒有資料。",
        tools=[my_docs],
    )
""")

VertexAiSearchTool 的建構參數:
  (self, *, data_store_id: 'Optional[str]' = None, data_store_specs: 'Optional[list[types.VertexAISearchDataStoreSpec]]' = None, search_engine_id: 'Optional[str]' = None, filter: 'Optional[str]' = None, max_results: 'Optional[int]' = None, bypass_multi_tools_limit: 'bool' = False)

用法（需要 Google Cloud 專案 + Vertex AI Search data store）：

    from google.adk.tools import VertexAiSearchTool

    my_docs = VertexAiSearchTool(
        data_store_id="projects/<PROJECT>/locations/<LOC>"
                      "/collections/default_collection/dataStores/<DATASTORE>"
    )

    agent = LlmAgent(
        name="doc_agent",
        model="gemini-flash-latest",
        instruction="依照公司內部文件回答，找不到就說沒有資料。",
        tools=[my_docs],
    )



### ⚠️ 這條路需要 GCP，AI Studio 的金鑰用不了

| | `google_search` | `VertexAiSearchTool` |
|---|---|---|
| 資料來源 | 公開網路 | **你自己的文件庫** |
| 認證 | AI Studio API key 就行 | **必須 GCP 專案 + ADC** |
| 顯示義務 | ✅ Search Suggestions | 無 |
| 免費層 | 有（額度很緊） | 無 |

本教材用的是 AI Studio 金鑰，所以 `VertexAiSearchTool` 這條**跑不起來**——
但程式碼是完整的，有 GCP 專案時直接可用。

## 7. ⚠️ 內建工具共存限制（呼應 Day 06）

再確認一次這個限制，因為它在 grounding 情境特別容易撞到：

In [8]:
def get_internal_price(sku: str) -> dict:
    """查詢內部報價。

    Args:
        sku: 商品編號。
    """
    return {"sku": sku, "price": 1280}


try:
    mixed = LlmAgent(
        name="mixed",
        model=get_model(),
        instruction="研究助理。",
        tools=[google_search, get_internal_price],
    )
    print(await run_once(mixed, "A-100 的報價是多少？"))
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:260]}")

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your 


### 📌 補充：有一個沒什麼人提的旁門

`VertexAiSearchTool` 的建構參數裡藏著一個 `bypass_multi_tools_limit`：

In [9]:
import inspect

sig = inspect.signature(VertexAiSearchTool.__init__)
print("VertexAiSearchTool 參數:")
for name, param in sig.parameters.items():
    if name == "self":
        continue
    mark = "  ← 就是這個" if name == "bypass_multi_tools_limit" else ""
    print(f"  {name} = {param.default}{mark}")

VertexAiSearchTool 參數:
  data_store_id = None
  data_store_specs = None
  search_engine_id = None
  filter = None
  max_results = None
  bypass_multi_tools_limit = False  ← 就是這個


設成 `True` 就能讓它跟其他工具並存，繞過那個限制。

但**它只存在於 `VertexAiSearchTool`**——`google_search` 沒有這個參數：

In [10]:
print("google_search 是:", type(google_search).__name__)
print("有 bypass_multi_tools_limit 嗎？",
      hasattr(google_search, "bypass_multi_tools_limit"))

google_search 是: GoogleSearchTool
有 bypass_multi_tools_limit 嗎？ True


所以這條旁門只對「自有資料 grounding」有用，而那條路需要 GCP。
**用 AI Studio 金鑰 + `google_search` 的人，還是只能走 `AgentTool`。**

想要「既能上網查、又能查內部資料」，標準解法是 `AgentTool`（Day 06 第 7 節）：

In [11]:
from google.adk.tools.agent_tool import AgentTool

web_researcher = LlmAgent(
    name="web_researcher",
    model=get_model(),
    description="用 Google 搜尋取得最新公開資訊並附上來源。",
    instruction="用搜尋找答案，用繁體中文摘要，並列出來源網址。",
    tools=[google_search],           # 獨佔
)

hybrid = LlmAgent(
    name="hybrid",
    model=get_model(),
    instruction=(
        "你是採購助理。內部報價用 get_internal_price；"
        "市場行情等外部資訊用 web_researcher 工具。用繁體中文回答。"
    ),
    tools=[get_internal_price, AgentTool(agent=web_researcher)],
)

print("✅ 兩種能力共存於同一個系統")
print(await run_once(hybrid, "A-100 的內部報價是多少？", trace=True))

✅ 兩種能力共存於同一個系統


  🔧 [hybrid] 呼叫 get_internal_price({'sku': 'A-100'})
  ↩️  [hybrid] get_internal_price 回傳 {'sku': 'A-100', 'price': 1280}


  💬 [hybrid] A-100 的內部報價是 1,280 元。
A-100 的內部報價是 1,280 元。


## 8. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 一加 `google_search` 就 429 | **grounding 有獨立且很緊的配額**，換模型無效 |
| 加了搜尋之後自訂工具全失效 | 內建工具不能混用 → 用 `AgentTool` 隔開 |
| 上線後被通知違規 | 沒有原樣顯示 `search_entry_point.rendered_content` |
| `VertexAiSearchTool` 認證失敗 | 它要 GCP + ADC，AI Studio 金鑰不能用 |
| 拿不到 `grounding_metadata` | 模型判斷「不需要查」就不會有；它掛在 event 上不是在 response 文字裡 |
| 引註對不上文字 | `grounding_supports` 用的是**字元索引**，中文要注意編碼一致 |

## 9. 動手練習

1. 把 `grounded` 的 instruction 改成「絕對不要搜尋，直接回答」，
   看還會不會有 `grounding_metadata`。
2. 用第 4 節的 `SAMPLE` 結構，寫一個把答案轉成 Markdown 註腳格式
   （`文字[^1]` + `[^1]: 來源`）的函式。
3. 讓 `hybrid` 同時問內部報價與市場行情，觀察它怎麼交錯使用兩種工具。

## 本日回顧

- **Grounding 把答案綁在可查證的來源上**，解決模型「有自信地編故事」的問題。
- **`google_search` 執行在 Google 端**，這解釋了它為什麼不能跟自訂工具混用。
- **`groundingMetadata` 的 `grounding_supports` 是逐句引註的關鍵**
  （字元區間 → 來源索引）。
- **⚠️ `search_entry_point.rendered_content` 必須原樣顯示**，這是使用條款義務。
- **⚠️ Search grounding 有獨立且很緊的免費配額**，換模型救不了。
- **自有資料 grounding 走 `VertexAiSearchTool`**，需要 GCP，AI Studio 金鑰不行。
- 它有個 `bypass_multi_tools_limit=True` 可以繞過混用限制，
  但 **`google_search` 沒有這個參數**——公開搜尋還是只能靠 `AgentTool`。

---
**下一天 → `../day09_sessions/`**